## 08 - XGBoost focus with CatBoost benchmark

**Data (no leakage):**
- **Source:** 01 feature engineering (is_online, distance_to_merchant, effective_distance, hour_of_day, day_of_week, tx_count_1h, tx_count_24h, amt_sum_24h, customer_age, + categoricals). Same 22 features for all models.
- **Split (02):** Time 80% train / 20% test (OOT) → `oot_train_index.csv`, `oot_test_index.csv`.
- **Cal/Eval:** Test first 50% = **cal** (used only for CatBoost calibration in 03). Test last 50% = **eval** (evaluation only; no model is trained or calibrated on eval).
- **Eval_val / Eval_test:** Split eval by **stratified** split (on is_fraud) so **eval_test is guaranteed to have fraud** samples. Eval_val = threshold + hyperparam selection only; Eval_test = final test set.

**Goal:** Train XGBoost on same train/features; tune hyperparams and select thresholds on eval_val. Report **final** metrics on **eval_test** (threshold-selection set ≠ test set). CatBoost (raw + calibrated) as benchmarks.

In [11]:
import json
import pandas as pd
import numpy as np
from pathlib import Path
import joblib
from sklearn.metrics import precision_score, recall_score, f1_score
from sklearn.preprocessing import OrdinalEncoder
from sklearn.model_selection import train_test_split
import xgboost as xgb

ROOT = Path.cwd()
MODEL_DIR = ROOT / "models" if (ROOT / "models" / "feature_config.json").exists() else ROOT / "fraud_detection" / "models"

with open(MODEL_DIR / "feature_config.json") as f:
    config = json.load(f)
feature_cols = config["feature_cols"]
cat_features = config["cat_features"]

train_df = pd.read_csv(MODEL_DIR / "oot_train_index.csv")
test_df = pd.read_csv(MODEL_DIR / "oot_test_index.csv")
X_test = test_df[feature_cols]
y_test = test_df["is_fraud"].values

n_cal = int(len(X_test) * 0.5)
X_eval = X_test.iloc[n_cal:].copy()
y_eval = y_test[n_cal:].copy()
n_eval = len(y_eval)
idx = np.arange(n_eval)
n_fraud_eval = int(y_eval.sum())
if n_fraud_eval >= 2:
    idx_val, idx_test = train_test_split(idx, test_size=0.5, random_state=42, stratify=y_eval)
else:
    idx_val, idx_test = train_test_split(idx, test_size=0.5, random_state=42)
    if n_fraud_eval == 1:
        fraud_idx = int(np.where(y_eval == 1)[0][0])
        if fraud_idx in idx_val:
            idx_val = np.setdiff1d(idx_val, [fraud_idx])
            idx_test = np.append(idx_test, fraud_idx)
assert int(y_eval[idx_test].sum()) >= 1 or n_fraud_eval == 0, "eval_test must have fraud when eval has fraud"
y_val = y_eval[idx_val]
y_test_hold = y_eval[idx_test]
n_val, n_test_hold = len(idx_val), len(idx_test)

print("Train:", len(train_df), "| Cal:", n_cal, "| Eval_val:", n_val, "| Eval_test:", n_test_hold)
print("Train fraud:", train_df["is_fraud"].sum(), "| Eval_val fraud:", int(y_val.sum()), "| Eval_test fraud:", int(y_test_hold.sum()))

Train: 444575 | Cal: 55572 | Eval_val: 27786 | Eval_test: 27786
Train fraud: 1961 | Eval_val fraud: 11 | Eval_test fraud: 10


In [12]:
def metrics_at_threshold(y_true, prob, th):
    pred = (prob >= th).astype(int)
    if pred.sum() == 0:
        return 0.0, 0.0, 0.0
    return precision_score(y_true, pred, zero_division=0), recall_score(y_true, pred, zero_division=0), f1_score(y_true, pred, zero_division=0)

def best_threshold_by_f1(y_true, prob, thresholds=None):
    thresholds = thresholds if thresholds is not None else np.linspace(0.05, 0.95, 37)
    best_f1, best_th = -1.0, 0.5
    best_prec, best_rec = 0.0, 0.0
    for th in thresholds:
        prec, rec, f1 = metrics_at_threshold(y_true, prob, th)
        if f1 > best_f1:
            best_f1, best_th, best_prec, best_rec = f1, th, prec, rec
    return best_th, best_prec, best_rec, best_f1

def row(y_true, pred, model_name, best_th):
    n_pred_pos = int(pred.sum())
    if n_pred_pos == 0:
        return {"Model": model_name, "best_th": best_th, "n_pred_pos": n_pred_pos, "Precision": 0.0, "Recall": 0.0, "F1": 0.0}
    p = precision_score(y_true, pred, zero_division=0)
    r = recall_score(y_true, pred, zero_division=0)
    f1 = f1_score(y_true, pred, zero_division=0)
    return {"Model": model_name, "best_th": best_th, "n_pred_pos": n_pred_pos, "Precision": round(p, 3), "Recall": round(r, 3), "F1": round(f1, 3)}

### CatBoost (benchmark): load from 02/03, predict on full eval

In [13]:
from catboost import CatBoostClassifier

base_cb = CatBoostClassifier()
base_cb.load_model(str(MODEL_DIR / "catboost_fraud.cbm"))
calibrated_cb = joblib.load(MODEL_DIR / "catboost_calibrated.pkl")

prob_raw = base_cb.predict_proba(X_eval)[:, 1]
prob_cal = calibrated_cb.predict_proba(X_eval)[:, 1]
prob_raw_val = prob_raw[idx_val]
prob_cal_val = prob_cal[idx_val]
print("CatBoost (raw + calibrated) predictions on full eval ready.")

CatBoost (raw + calibrated) predictions on full eval ready.


### XGBoost: same 22 features, preprocessing (label encode + fillna), train on train only; optional tune on eval_val

In [14]:
X_train_cb = train_df[feature_cols]
y_train = train_df["is_fraud"].astype(int).values

enc = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
X_train_enc = X_train_cb.copy()
X_eval_enc = X_eval.copy()
if cat_features:
    enc.fit(X_train_cb[cat_features].astype(str).fillna("__NA__"))
    X_train_enc[cat_features] = enc.transform(X_train_cb[cat_features].astype(str).fillna("__NA__"))
    X_eval_enc[cat_features] = enc.transform(X_eval[cat_features].astype(str).fillna("__NA__"))
numeric_cols = [c for c in feature_cols if c not in cat_features]
for c in numeric_cols:
    X_train_enc[c] = pd.to_numeric(X_train_enc[c], errors="coerce")
    X_eval_enc[c] = pd.to_numeric(X_eval_enc[c], errors="coerce")
med = X_train_enc[numeric_cols].median()
X_train_enc[numeric_cols] = X_train_enc[numeric_cols].fillna(med)
X_eval_enc[numeric_cols] = X_eval_enc[numeric_cols].fillna(med)
X_train_xgb = X_train_enc[feature_cols].values.astype(np.float64)
X_eval_xgb = X_eval_enc[feature_cols].values.astype(np.float64)

X_tr, X_val_inner, y_tr, y_val_inner = train_test_split(X_train_xgb, y_train, test_size=0.1, random_state=42, stratify=y_train)
neg, pos = (y_tr == 0).sum(), max((y_tr == 1).sum(), 1)
base_scale = neg / pos

best_f1_val, best_spw, best_depth = -1.0, base_scale, 6
for scale_pos_weight in [base_scale * 0.8, base_scale, base_scale * 1.2]:
    for max_depth in [5, 6, 7]:
        clf = xgb.XGBClassifier(n_estimators=500, max_depth=max_depth, learning_rate=0.05, scale_pos_weight=scale_pos_weight,
                               random_state=42, eval_metric="auc", use_label_encoder=False)
        clf.fit(X_tr, y_tr, eval_set=[(X_val_inner, y_val_inner)], verbose=False)
        prob_val = clf.predict_proba(X_eval_xgb[idx_val])[:, 1]
        th, pr, rec, f1 = best_threshold_by_f1(y_val, prob_val)
        if f1 > best_f1_val:
            best_f1_val, best_spw, best_depth = f1, scale_pos_weight, max_depth

xgb_clf = xgb.XGBClassifier(n_estimators=500, max_depth=best_depth, learning_rate=0.05, scale_pos_weight=best_spw,
                            random_state=42, eval_metric="auc", use_label_encoder=False)
xgb_clf.fit(X_tr, y_tr, eval_set=[(X_val_inner, y_val_inner)], verbose=False)
prob_xgb = xgb_clf.predict_proba(X_eval_xgb)[:, 1]
prob_xgb_val = prob_xgb[idx_val]
print("XGBoost trained on train only; best scale_pos_weight=%.0f, max_depth=%d (by F1 on eval_val)." % (best_spw, best_depth))

/Users/zhumiban/anaconda3/envs/agent_bank/lib/python3.10/site-packages/xgboost/training.py:200: UserWarning: [12:45:46] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/Users/zhumiban/anaconda3/envs/agent_bank/lib/python3.10/site-packages/xgboost/training.py:200: UserWarning: [12:45:57] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/Users/zhumiban/anaconda3/envs/agent_bank/lib/python3.10/site-packages/xgboost/training.py:200: UserWarning: [12:46:10] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/Users/zhumiban/anaconda3/envs/agent_bank/lib/python3.10/site-packages/xgboost/training.py:200: UserWarning: [12:46:24] WARNING: /Users/runner/work/xgboost/xgbo

XGBoost trained on train only; best scale_pos_weight=226, max_depth=7 (by F1 on eval_val).


### Threshold optimization on eval_val; report on full eval

In [15]:
th_raw, _, _, _ = best_threshold_by_f1(y_val, prob_raw_val)
th_cal, _, _, _ = best_threshold_by_f1(y_val, prob_cal_val)
th_xgb, _, _, _ = best_threshold_by_f1(y_val, prob_xgb_val)

prob_raw_test = prob_raw[idx_test]
prob_cal_test = prob_cal[idx_test]
prob_xgb_test = prob_xgb[idx_test]
pred_raw_test = (prob_raw_test >= th_raw).astype(int)
pred_cal_test = (prob_cal_test >= th_cal).astype(int)
pred_xgb_test = (prob_xgb_test >= th_xgb).astype(int)

final = pd.DataFrame([
    row(y_test_hold, pred_raw_test, "Raw CatBoost", th_raw),
    row(y_test_hold, pred_cal_test, "Calibrated CatBoost", th_cal),
    row(y_test_hold, pred_xgb_test, "XGBoost", th_xgb),
])
print("Final test = eval_test (n=%d, n_fraud=%d). Thresholds chosen on eval_val only." % (n_test_hold, int(y_test_hold.sum())))
display(final)

final_full = pd.DataFrame([
    row(y_eval, (prob_raw >= th_raw).astype(int), "Raw CatBoost", th_raw),
    row(y_eval, (prob_cal >= th_cal).astype(int), "Calibrated CatBoost", th_cal),
    row(y_eval, (prob_xgb >= th_xgb).astype(int), "XGBoost", th_xgb),
])
print("Reference (same thresholds on full eval, n_fraud=%d):" % int(y_eval.sum()))
display(final_full)

Final test = eval_test (n=27786, n_fraud=10). Thresholds chosen on eval_val only.


,Model,best_th,n_pred_pos,Precision,Recall,F1
0,Raw CatBoost,0.95,8,0.750,0.6,0.667
1,Calibrated CatBoost,0.80,8,0.750,0.6,0.667
2,XGBoost,0.95,9,0.778,0.7,0.737


Reference (same thresholds on full eval, n_fraud=21):


,Model,best_th,n_pred_pos,Precision,Recall,F1
0,Raw CatBoost,0.95,20,0.700,0.667,0.683
1,Calibrated CatBoost,0.80,18,0.778,0.667,0.718
2,XGBoost,0.95,19,0.842,0.762,0.800


### Save XGBoost model and preprocessing to ml_model

Saves to `fraud_detection/ml_model/`: XGBoost model, OrdinalEncoder, numeric medians, feature config, and best threshold (for inference).

In [16]:
ML_MODEL_DIR = (ROOT / "fraud_detection" / "ml_model") if (ROOT / "fraud_detection").is_dir() else (ROOT / "ml_model")
ML_MODEL_DIR.mkdir(parents=True, exist_ok=True)

joblib.dump(xgb_clf, ML_MODEL_DIR / "xgboost_fraud.pkl")

numeric_median_dict = med.to_dict()
preprocess_and_config = {
    "encoder": enc,
    "numeric_median": numeric_median_dict,
    "feature_cols": feature_cols,
    "cat_features": cat_features,
    "threshold": float(th_xgb),
    "scale_pos_weight": float(best_spw),
    "max_depth": int(best_depth),
}
joblib.dump(preprocess_and_config, ML_MODEL_DIR / "preprocess_and_config.pkl")

import shutil
shutil.copy(MODEL_DIR / "feature_config.json", ML_MODEL_DIR / "feature_config.json")

print("Saved to", ML_MODEL_DIR)
print("  xgboost_fraud.pkl, preprocess_and_config.pkl, feature_config.json")
print("  threshold =", th_xgb, "| scale_pos_weight =", best_spw, "| max_depth =", best_depth)

Saved to /Users/zhumiban/Desktop/agent_bank/Banking_Agent_reorg/fraud_detection/ml_model
  xgboost_fraud.pkl, preprocess_and_config.pkl, feature_config.json
  threshold = 0.95 | scale_pos_weight = 225.69518413597734 | max_depth = 7
